<a href="https://colab.research.google.com/github/hwasun-zip/Behind-the-Coupon/blob/main/notebooks/03_stress_test.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [10]:
# 작업 환경 복구
from google.colab import drive
drive.mount('/content/drive')

!pip install -q koreanize-matplotlib
import os, sys
import numpy as np, pandas as pd, matplotlib.pyplot as plt
import koreanize_matplotlib
plt.rcParams['axes.unicode_minus'] = False

BASE = '/content/drive/MyDrive/Behind-the-Coupon'
sys.path.append(f'{BASE}/src')
from els_engine import *

raw = pd.read_csv(f'{BASE}/data/index_close.csv', index_col=0, parse_dates=True)
px = raw.sort_index().ffill()
print(px.shape, px.index.min().date(), '~', px.index.max().date())

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
(5921, 5) 2004-01-02 ~ 2026-10-02


In [11]:
# 구조, 조합, 시나리오 설정
STRUCT = dict(strikes=(0.90, 0.90, 0.85, 0.85, 0.80, 0.75), ki=0.50)

COMBOS = {
    'S&P+유로+H지수':    ['SPX', 'SX5E', 'HSCEI'],
    'S&P+유로+니케이':   ['SPX', 'SX5E', 'NKY'],
    'KOSPI200+S&P+유로': ['KOSPI200', 'SPX', 'SX5E'],
    'S&P+유로 (2개)':    ['SPX', 'SX5E'],
    'S&P 단독':          ['SPX'],
}

SCENARIOS = {
    '2008 금융위기':    '2007-10-31',
    '2015 중국 쇼크':   '2015-05-26',
    '2020 코로나':      '2020-02-19',
    '2021 H지수 고점':  '2021-02-17',
    '2022 긴축 하락장': '2022-01-03',
}

In [12]:
# 위기 직전 발행 시 결과 (하루씩 발행한 결과)
def stress_one(assets, start, **kw):
    """특정 날짜(휴일이면 그 직후 첫 거래일)에 발행된 ELS 1개의 결과"""
    s = pd.Timestamp(start)
    r = simulate_stepdown(px, assets, start=s, end=s + pd.Timedelta(days=10), **kw)
    return r.iloc[0]

rows = []
for sc, day in SCENARIOS.items():
    for name, assets in COMBOS.items():
        r = stress_one(assets, day, **STRUCT)
        rows.append({
            '시나리오': sc, '조합': name,
            '결과': r['outcome'],
            '상환시점(년)': r['years'],
            '원금회수': r['principal'],
            '보유중최저': r['worst_min'],
        })
stress = pd.DataFrame(rows)
order = list(COMBOS)

print('[결과]')
display(stress.pivot(index='조합', columns='시나리오', values='결과').loc[order])
print('[보유 중 최악 지수의 최저점 (발행일=1)]')
display(stress.pivot(index='조합', columns='시나리오', values='보유중최저').loc[order].round(2))

[결과]


시나리오,2008 금융위기,2015 중국 쇼크,2020 코로나,2021 H지수 고점,2022 긴축 하락장
조합,,,,,
S&P+유로+H지수,손실,만기상환(행사가),조기상환,손실,만기상환(행사가)
S&P+유로+니케이,손실,조기상환,조기상환,조기상환,조기상환
KOSPI200+S&P+유로,손실,조기상환,조기상환,조기상환,조기상환
S&P+유로 (2개),손실,조기상환,조기상환,조기상환,조기상환
S&P 단독,만기상환(행사가),조기상환,조기상환,조기상환,조기상환


[보유 중 최악 지수의 최저점 (발행일=1)]


시나리오,2008 금융위기,2015 중국 쇼크,2020 코로나,2021 H지수 고점,2022 긴축 하락장
조합,,,,,
S&P+유로+H지수,0.25,0.51,0.62,0.40,0.60
S&P+유로+니케이,0.40,0.83,0.62,0.90,0.75
KOSPI200+S&P+유로,0.40,0.83,0.62,0.95,0.71
S&P+유로 (2개),0.40,0.83,0.62,0.96,0.75
S&P 단독,0.44,0.89,0.66,0.96,0.75


In [13]:
# 위기 전후 ±3개월 발행분 전체로 확인
win = []
for sc, day in SCENARIOS.items():
    day = pd.Timestamp(day)
    for name, assets in COMBOS.items():
        r = simulate_stepdown(px, assets,
                              start=day - pd.DateOffset(months=3),
                              end=day + pd.DateOffset(months=3), **STRUCT)
        loss = r['outcome'] == '손실'
        win.append({
            '시나리오': sc, '조합': name,
            '발행건수': len(r),
            '손실확률': loss.mean(),
            '손실시평균손실률': 1 - r.loc[loss, 'principal'].mean() if loss.any() else 0.0,
            '최악원금회수': r['principal'].min(),
        })
win = pd.DataFrame(win)

print('[위기 전후 6개월 발행분의 손실확률]')
display(win.pivot(index='조합', columns='시나리오', values='손실확률').loc[order].round(2))
print('[최악의 경우 원금 회수율]')
display(win.pivot(index='조합', columns='시나리오', values='최악원금회수').loc[order].round(2))

# 체계적 위기(2008)에선 분산이 안 통하고, 고유 위기(2021)에선 분산이 통한다
# 손실은 "깊고 오래가는" 위기에서만 났다 (5개 중 2개)
# 최악의 경우 원금의 40~58% 손실 → 판매 한도와 집중도 관리의 근거

[위기 전후 6개월 발행분의 손실확률]


시나리오,2008 금융위기,2015 중국 쇼크,2020 코로나,2021 H지수 고점,2022 긴축 하락장
조합,,,,,
S&P+유로+H지수,0.86,0.0,0.0,0.71,0.0
S&P+유로+니케이,0.90,0.0,0.0,0.00,0.0
KOSPI200+S&P+유로,0.83,0.0,0.0,0.00,0.0
S&P+유로 (2개),0.83,0.0,0.0,0.00,0.0
S&P 단독,0.11,0.0,0.0,0.00,0.0


[최악의 경우 원금 회수율]


시나리오,2008 금융위기,2015 중국 쇼크,2020 코로나,2021 H지수 고점,2022 긴축 하락장
조합,,,,,
S&P+유로+H지수,0.60,1.0,1.0,0.42,1.0
S&P+유로+니케이,0.53,1.0,1.0,1.00,1.0
KOSPI200+S&P+유로,0.60,1.0,1.0,1.00,1.0
S&P+유로 (2개),0.60,1.0,1.0,1.00,1.0
S&P 단독,0.74,1.0,1.0,1.00,1.0


In [14]:
# 결과 저장
stress.to_csv(f'{BASE}/output/03_stress_single_day.csv', index=False)
win.round(4).to_csv(f'{BASE}/output/03_stress_window.csv', index=False)
print('저장 완료')

저장 완료


In [15]:
# 마지막 분석: 보유 중 모니터링 (스트레스 테스트가 발행 전 대비라면, 이건 발행 후 관리)
# 이미 녹인을 맞은 ELS는 다 손실일까? 녹인을 언제 맞았냐에 따라 회복 가능성이 다를까?
# -> 발행 초반에 녹인을 맞으면 만기까지 남은 시간이 많아서 회복할 기회가 있고, 만기 직전에 맞으면 회복할 시간이 없어서 손실이 될 가능성이 클 것 같다는 가정

def ki_touch_timing(assets, res, ki=0.50, obs_months=6):
    """녹인을 맞은 ELS별로 '발행 후 몇 개월째에 처음 맞았는지' 계산"""
    p = px[assets].dropna()
    dates, vals = p.index, p.values
    out = []
    for _, row in res[res['ki_touch']].iterrows():
        i0 = dates.get_loc(row['issue_date'])
        end_day = row['issue_date'] + pd.DateOffset(months=int(row['redeem_step'] * obs_months))
        i_end = dates.searchsorted(end_day)
        worst = (vals[i0:i_end + 1] / vals[i0]).min(axis=1)
        first = int(np.argmax(worst < ki))                     # 처음 녹인 아래로 내려간 날
        months_in = (dates[i0 + first] - row['issue_date']).days / 30.44
        out.append({
            'issue_date': row['issue_date'],
            '녹인터치(발행후개월)': months_in,
            '손실': int(row['outcome'] == '손실'),
        })
    return pd.DataFrame(out)

parts = []
for name, assets in COMBOS.items():
    r = simulate_stepdown(px, assets, start='2007-03-30', **STRUCT)
    t = ki_touch_timing(assets, r)
    t['조합'] = name
    parts.append(t)
touch = pd.concat(parts, ignore_index=True)

print(f'녹인 터치 ELS: {len(touch):,}건, 그중 최종 손실 {touch["손실"].mean():.1%}\n')

bins = [0, 6, 12, 18, 24, 30, 36]
labels = ['0~6개월', '6~12', '12~18', '18~24', '24~30', '30~36']
touch['터치시점'] = pd.cut(touch['녹인터치(발행후개월)'], bins=bins, labels=labels)
touch.groupby('터치시점', observed=True).agg(
    건수=('손실', 'size'),
    최종손실확률=('손실', 'mean'),
).round(3)

녹인 터치 ELS: 1,197건, 그중 최종 손실 67.1%



,건수,최종손실확률
터치시점,,
0~6개월,111,0.180
6~12,472,0.568
12~18,520,0.812
18~24,94,0.989


In [16]:
# 저장
touch.to_csv(f'{BASE}/output/03_ki_touch_timing.csv', index=False)
print('저장 완료')

저장 완료
